# Laboratório: embeddings de textos

Vamos transformar textos (notícias, filmes e séries da Netflix ou jogos de videogame, à sua escolha) em vetores de números e usar esses
vetores de duas formas: **agrupar textos parecidos** e **buscar pelo sentido**, não só pelas palavras.

- Ler os vetores (embeddings) e comparar dois textos pelo cosseno.
- Ver o mapa dos textos e os grupos que aparecem.
- Buscar pelo sentido, com um banco de vetores.
- Dois desafios: recomendar por conteúdo e misturar dois gostos.

Cada passo tem uma **Ideia** (o que estamos fazendo e por quê) e uma **Dica** (qual função usar). As
linhas que você escreve estão marcadas com um comentário que começa com `#`; as outras já vêm prontas.

## 1. Preparação

**Ideia:** o notebook roda no **Google Colab** ou no seu computador. Instalamos as bibliotecas (uma vez),
importamos tudo e deixamos prontas as funções que não são o foco da aula.

**Dica:** só rode as células. A célula "Funções prontas" é longa de propósito: você usa as funções, não
precisa escrever nenhuma.

In [ ]:
%pip install -q sentence-transformers zvec umap-learn plotly


In [ ]:
import tempfile
import urllib.request
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")  # esconde avisos das bibliotecas, que só atrapalham a leitura

import numpy as np
import pandas as pd
import plotly.express as px
import zvec
from sentence_transformers import SentenceTransformer
from sklearn.cluster import KMeans
from umap import UMAP

SEMENTE = 42  # repete sempre o mesmo sorteio
LIGHT = "#cbd5e1"

pd.set_option("display.max_colwidth", 90)


Escolha o conjunto de dados (troque o texto entre aspas e rode as células de novo, do começo):
`"noticias"`, `"netflix"` ou `"jogos"`.

In [ ]:
DATASET = "noticias"   # escolha: "noticias", "netflix" ou "jogos"


**Funções prontas:**

In [ ]:
# ---------------------------------------------------------------------------------------------
# Funções prontas (você só usa; não precisa escrever nenhuma)
# ---------------------------------------------------------------------------------------------
URL_DADOS = "https://raw.githubusercontent.com/chcomin/curso-processamento-e-visualizacao-de-dados/main/M08_embeddings_de_texto/laboratorio/dados_m08/"
PASTA = Path("dados_m08")  # Altere para o caminho do seu sistema, se já tiver os arquivos

# A instrução diz ao modelo qual é a tarefa da busca (só a consulta leva; os textos não)
INSTRUCOES = {
    "b2w": "Given a retail product title, retrieve titles of similar products",
    "noticias": "Given a search query, retrieve relevant news articles",
    "netflix": "Given a short description, retrieve movies and TV shows whose plot matches it",
    "jogos": "Given a short description, retrieve video games that match it",
}

modelos = {}  # guarda o modelo depois de carregado (carregar leva alguns segundos)


def carregar_corpus(nome):
    """Lê a tabela (titulo, texto, categoria) e os vetores prontos; baixa do GitHub se não estiverem na pasta."""
    PASTA.mkdir(exist_ok=True)
    for arquivo in [nome + ".csv", nome + "_vetores.npy"]:
        if not (PASTA / arquivo).exists():
            print("baixando", arquivo, "...")
            urllib.request.urlretrieve(URL_DADOS + arquivo, PASTA / arquivo)
    tabela = pd.read_csv(PASTA / (nome + ".csv"))
    X = np.load(PASTA / (nome + "_vetores.npy")).astype("float32")  # os vetores vêm em float16, que ocupa menos
    X = X / np.linalg.norm(X, axis=1, keepdims=True)  # garante comprimento 1 (o float16 perde um pouco)
    return tabela, X


def pegar_modelo():
    """O modelo Qwen3 (600 milhões de parâmetros): baixa na primeira vez e depois só reaproveita."""
    if "qwen" not in modelos:
        modelos["qwen"] = SentenceTransformer("Qwen/Qwen3-Embedding-0.6B")
    return modelos["qwen"]


def vetorizar(textos):
    """Vetoriza QUALQUER lista de textos com o Qwen3: devolve uma matriz (n textos, dimensão 1024).

    Exemplo com o seu próprio arquivo (precisa de uma coluna "titulo" e outra "categoria" para o resto):
        meus = pd.read_csv("meus_textos.csv")
        X_meus = vetorizar(meus["texto"])
    """
    modelo = pegar_modelo()
    return modelo.encode(list(textos), batch_size=32, normalize_embeddings=True, show_progress_bar=True)


def vetorizar_consulta(texto):
    """Vetoriza uma consulta de busca (com a instrução do conjunto de dados escolhido)."""
    prompt = "Instruct: " + INSTRUCOES[DATASET] + "\nQuery:"
    return pegar_modelo().encode(texto, prompt=prompt, normalize_embeddings=True)


def mapa(tabela, xy, cor):
    """Mapa 2D: um ponto por texto, passe o mouse para ler o título.

    `cor` é o nome de uma coluna da tabela (ex.: "categoria") ou a lista de grupos do KMeans.
    """
    if isinstance(cor, str):
        valores = tabela[cor].astype(str).reset_index(drop=True)
    else:
        valores = "grupo " + pd.Series(cor).astype(str)
    maiores = valores.value_counts().head(10).index  # no máximo 10 cores; o resto vira "Outras"
    df = pd.DataFrame({"x": xy[:, 0], "y": xy[:, 1], "cor": valores.where(valores.isin(maiores), "Outras"),
                       "titulo": tabela["titulo"].str.slice(0, 120).values}).sort_values("cor")
    figura = px.scatter(df, x="x", y="y", color="cor", hover_name="titulo",
                        hover_data={"x": False, "y": False, "cor": False},
                        color_discrete_map={"Outras": LIGHT},
                        color_discrete_sequence=px.colors.qualitative.T10,
                        width=900, height=600)
    figura.update_traces(marker_size=4)
    figura.update_xaxes(visible=False)
    figura.update_yaxes(visible=False)
    figura.update_layout(plot_bgcolor="white", legend_title_text="")
    figura.show()


def resumo(tabela, linha):
    """Uma linha com o título e o começo do texto do item `linha` (sem repetir o título)."""
    titulo = tabela["titulo"].iloc[linha]
    detalhe = tabela["texto"].iloc[linha]
    if detalhe.startswith(titulo):
        detalhe = detalhe[len(titulo):]  # tira o título do começo do texto
    detalhe = detalhe.strip(" .")
    if detalhe:
        return titulo[:50] + " | " + detalhe[:80]
    return titulo[:90]


def mostrar_grupos(tabela, grupos, n=5):
    """Para cada grupo: tamanho, categoria mais comum e `n` exemplos (título e começo do texto)."""
    grupos = np.asarray(grupos)
    for grupo in sorted(set(grupos)):
        do_grupo = tabela[grupos == grupo]
        categoria = do_grupo["categoria"].value_counts().index[0]
        print(f"Grupo {grupo}: {len(do_grupo)} textos; categoria mais comum: {categoria}")
        for linha in do_grupo.sample(min(n, len(do_grupo)), random_state=SEMENTE).index:
            print("    ", resumo(tabela, linha))
        print()


def criar_colecao(tabela, X):
    """Cria uma coleção zvec (banco de vetores) numa pasta temporária e insere todos os textos."""
    pasta = tempfile.mkdtemp() + "/colecao"
    indice = zvec.HnswIndexParam(metric_type=zvec.MetricType.COSINE)  # compara por cosseno
    vetor = zvec.VectorSchema(name="embedding", data_type=zvec.DataType.VECTOR_FP32,
                              dimension=X.shape[1], index_param=indice)
    campos = [zvec.FieldSchema(name="linha", data_type=zvec.DataType.INT32),
              zvec.FieldSchema(name="titulo", data_type=zvec.DataType.STRING),
              zvec.FieldSchema(name="categoria", data_type=zvec.DataType.STRING),
              zvec.FieldSchema(name="texto", data_type=zvec.DataType.STRING)]
    colecao = zvec.create_and_open(path=pasta, schema=zvec.CollectionSchema(name="textos", fields=campos, vectors=[vetor]))

    docs = []
    for linha in range(len(tabela)):
        campos_do_texto = {"linha": linha, "titulo": str(tabela["titulo"].iloc[linha]),
                           "categoria": str(tabela["categoria"].iloc[linha]), "texto": str(tabela["texto"].iloc[linha])}
        docs.append(zvec.Doc(id=str(linha), vectors={"embedding": X[linha].tolist()}, fields=campos_do_texto))

    for inicio in range(0, len(docs), 1000):  # o zvec aceita até 1024 documentos por vez
        colecao.insert(docs[inicio:inicio + 1000])
    colecao.optimize()  # monta o índice de busca
    return colecao


def buscar(colecao, vetor, k=5):
    """Os `k` textos mais próximos do vetor, em uma tabela (o índice é a linha do texto na tabela)."""
    resultado = colecao.query(zvec.Query(field_name="embedding", vector=vetor.tolist()), topk=k)
    linhas = []
    for item in resultado:
        similaridade = 1 - item.score  # o zvec devolve a distância de cosseno; a similaridade é 1 - distância
        campos = item.fields
        linhas.append([int(campos["linha"]), campos["titulo"], campos["categoria"], round(similaridade, 3), campos["texto"]])
    resultado = pd.DataFrame(linhas, columns=["linha", "titulo", "categoria", "similaridade", "texto"]).set_index("linha")
    if (resultado["texto"] == resultado["titulo"]).all():
        resultado = resultado.drop(columns="texto")  # quando o texto é só o título, não repete
    return resultado


## 2. Carregar a amostra

**Ideia:** um **embedding** é uma lista de números que um modelo neural atribui a um texto: textos de
sentido parecido recebem listas parecidas. Aqui usamos 3.000 textos e os vetores já calculados (um vetor de
dimensão 1024 por texto), alinhados linha a linha com a tabela.

**Dica:** `carregar_corpus(DATASET)` devolve a tabela e a matriz de vetores `X`. Com
`VETORIZAR_AQUI = True` o modelo calcula os vetores agora (leva minutos; no Colab ligue a GPU em
*Ambiente de execução*). Deixe `False` para ir rápido.

In [ ]:
# carregue a tabela e os vetores com a função pronta: tabela, X = carregar_corpus(DATASET)

VETORIZAR_AQUI = False  # False: usa os vetores já calculados. True: calcula agora com o modelo
if VETORIZAR_AQUI:
    print("calculando os vetores agora...")
    X = vetorizar(tabela["texto"])

print("tabela:", tabela.shape)
print("vetores:", X.shape)
tabela.head()


## 3. Um vetor e o cosseno

**Ideia:** cada texto é uma linha de `X`: um vetor de dimensão 1024. Para comparar dois textos, comparamos
os vetores deles.

**Similaridade de cosseno.** Mede o quanto dois vetores $u$ e $v$ apontam na mesma direção: é o cosseno
do ângulo $\theta$ entre eles.

$$\cos(u, v) = \frac{u \cdot v}{\lVert u \rVert \, \lVert v \rVert}
= \frac{\sum_{i=1}^{1024} u_i \, v_i}{\sqrt{\sum_{i=1}^{1024} u_i^2} \, \sqrt{\sum_{i=1}^{1024} v_i^2}}$$

- $u \cdot v$ é o **produto escalar**: multiplica os vetores número a número e soma tudo;
- $\lVert u \rVert$ é o **comprimento** do vetor $u$.

**Como ler:** o cosseno vai de −1 a 1. Perto de **1**, os vetores apontam na mesma direção: sentido
parecido. Perto de **0**, são perpendiculares: assuntos sem relação. Em **−1**, apontam em direções
opostas. O cosseno não depende do comprimento dos vetores, só da direção.

**Atalho:** os vetores aqui já têm comprimento 1 (estão normalizados). Então o denominador vale 1 e o
cosseno é só o produto escalar.

**Dica:** o produto escalar de dois vetores `a` e `b` é `a @ b`. Compare um par parecido com um par
diferente.

In [ ]:
print("primeiros números do vetor do texto 0:", X[0][:10])

# posições na tabela: i e j são parecidos, k é de outro assunto
PARES = {"b2w": (104, 2143, 2263), "noticias": (432, 2012, 615), "netflix": (2849, 1773, 1672), "jogos": (432, 2244, 957)}
i, j, k = PARES[DATASET]
for posicao in [i, j, k]:
    print(posicao, resumo(tabela, posicao))

# cosseno do par parecido (produto escalar): cosseno_parecido = X[i] @ X[j]

# cosseno do par diferente: cosseno_diferente = X[i] @ X[k]

print(f"\nparecidos:  {cosseno_parecido:.2f}")
print(f"diferentes: {cosseno_diferente:.2f}")


## 4. Da dimensão 1024 para 2: o mapa

**Ideia:** uma tela só mostra 2 dimensões. O **UMAP** projeta os vetores em 2D mantendo os vizinhos
próximos. Cuidado: a distância entre grupos distantes no mapa não vale; o mapa sugere hipóteses, não
prova nada.

**Dica:** `UMAP(n_components=2, metric="cosine", random_state=SEMENTE)` cria o redutor e
`.fit_transform(X)` devolve a posição (x, y) de cada texto. Depois, `mapa(tabela, xy, "categoria")`
(pronto) desenha. Passe o mouse nos pontos: **que categorias ficam juntas? Quais se misturam?**

In [ ]:
# a posição 2D de cada texto: xy = UMAP(n_components=2, metric="cosine", random_state=SEMENTE).fit_transform(X)

mapa(tabela, xy, "categoria")


## 5. Grupos de textos parecidos

**Ideia:** o **KMeans** agrupa os vetores em 8 grupos, sem olhar para a categoria. Veja se os grupos
coincidem com as categorias e dê um nome a alguns.

**Dica:** `KMeans(n_clusters=8, random_state=SEMENTE, n_init="auto")` cria o agrupador e
`.fit_predict(X)` devolve o grupo de cada texto. O mapa por grupo e a lista de exemplos já estão prontos.

In [ ]:
# o KMeans com 8 grupos: kmeans = KMeans(n_clusters=8, random_state=SEMENTE, n_init="auto")

# o grupo de cada texto: grupos = kmeans.fit_predict(X)

mapa(tabela, xy, grupos)


Leia o tamanho, a categoria mais comum e os exemplos de cada grupo. Alguns grupos são claros; outros
são uma mistura, e isso também é uma resposta. (Se você recalcular os vetores com
`VETORIZAR_AQUI = True`, os grupos mudam um pouco: dê os nomes pela sua saída.)

In [ ]:
mostrar_grupos(tabela, grupos)


In [ ]:
# dê um nome a 3 grupos, olhando a saída acima: nomes = {0: "...", 3: "...", 5: "..."}

print(nomes)


## 6. Busca semântica com um banco de vetores

**Ideia:** o **zvec** guarda os vetores e acha, rápido, os mais próximos de uma consulta. A consulta
também vira vetor (com o mesmo modelo) e o banco devolve os textos de vetor mais próximo: a busca é pelo
**sentido**, a consulta não precisa ter as palavras do texto. A similaridade é o cosseno.

**Dica:** `criar_colecao(tabela, X)` monta o banco; `vetorizar_consulta("texto")` transforma a consulta
em vetor; `buscar(colecao, q)` devolve os 5 mais próximos. A primeira consulta baixa o modelo
(cerca de 1 GB, uma vez). Sugestões de consulta:

- notícias: *"eleição para presidente"* ou *"crise no futebol brasileiro"*
- filmes e séries da Netflix: *"crianças em uma aventura mágica"* ou *"documentário sobre a natureza"*
- jogos de videogame: *"jogo de corrida de carros"* ou *"jogo de terror em uma casa abandonada"*

In [ ]:
# crie o banco de vetores com a função pronta: colecao = criar_colecao(tabela, X)

print(colecao.stats)


In [ ]:
# vetorize a consulta (uma das sugestões acima): q = vetorizar_consulta("texto da consulta")

# busque no banco: buscar(colecao, q)


**Sua vez:** escreva uma consulta sua, descrevendo o que você procura com outras palavras. Os resultados
fazem sentido? Tente uma consulta que não tenha nenhuma palavra em comum com o texto que você espera achar.

In [ ]:
# vetorize a sua consulta: q = vetorizar_consulta("...")

# busque no banco: buscar(colecao, q)


## 7. Desafio 1: recomendação por conteúdo

**Ideia:** "a pessoa gostou deste item; o que mais recomendar?" é achar os vizinhos mais próximos do
vetor dele. Aqui a consulta não é um texto: já é um vetor, o do próprio item (sem instrução).

**Dica:** use `X[i]` como consulta e peça 6 resultados (`k=6`): o primeiro é o próprio item. A linha pronta
`resultado.drop(index=i)` tira esse item, sobrando 5 recomendações. Troque `i` por outro item.

In [ ]:
GOSTOU = {"noticias": 1417, "netflix": 643, "jogos": 75}
i = GOSTOU[DATASET]  # troque por outra posição da tabela
print("A pessoa gostou de:", resumo(tabela, i))

# o vetor do próprio item é a consulta: q = X[i]

# os 6 mais próximos: resultado = buscar(colecao, q, k=6)

resultado.drop(index=i)  # tira o próprio item


## 8. Desafio 2: gostei de dois itens

**Ideia:** vetores permitem **operações**, como na aula (`rei - homem + mulher`). A **média** de dois
vetores cai "no meio" dos dois itens: a busca acha o que lembra os dois ao mesmo tempo. Depois da média o
vetor fica mais curto que 1; dividimos pelo comprimento para voltar a 1.

**Dica:** `(X[i] + X[j]) / 2` é a média. `np.linalg.norm(q)` é o comprimento de `q`. Peça 7 resultados;
os dois itens originais costumam vir no topo, e a linha pronta os tira.

In [ ]:
DOIS = {"noticias": (1533, 1417), "netflix": (643, 2849), "jogos": (2544, 1998)}
i, j = DOIS[DATASET]  # troque por outras posições da tabela
print("A pessoa gostou de:")
print("  ", resumo(tabela, i))
print("  ", resumo(tabela, j))

# a média dos dois vetores: q = (X[i] + X[j]) / 2

# volte o comprimento para 1: q / np.linalg.norm(q)

# os 7 mais próximos: resultado = buscar(colecao, q, k=7)

resultado.drop(index=[i, j], errors="ignore")  # tira os dois itens originais


## 9. Fechamento

- Um modelo transforma textos em vetores; "perto no espaço" quer dizer "parecido no sentido", medido pelo
  cosseno.
- O mapa 2D (UMAP) e os grupos (KMeans) mostram quais textos se parecem, sem olhar para as categorias.
- A busca semântica acha textos pelo sentido da consulta; recomendar é buscar com o vetor de um item e misturar gostos é fazer uma média de vetores.